### Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

#### Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
 - Answer relevance
 - Answer accuracy
 - Retrieval quality
 
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

### Chatbot Evaluation

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [2]:
from langsmith import Client

client = Client()

# Generic FAQ pairs — gold answers are full sentences so they match how the chatbot replies
dataset_name = "Chatbot FAQ Evaluation"
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name)
    client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {
                "inputs": {"question": "What is Python?"},
                "outputs": {"answer": "Python is a popular programming language used for web, data, and AI applications."},
            },
            {
                "inputs": {"question": "What is an API?"},
                "outputs": {"answer": "An API is a way for two software systems to communicate with each other."},
            },
            {
                "inputs": {"question": "What is Git?"},
                "outputs": {"answer": "Git is a version control system that tracks changes in source code."},
            },
            {
                "inputs": {"question": "What is a large language model?"},
                "outputs": {"answer": "A large language model is an AI model trained on text to understand and generate language."},
            },
            {
                "inputs": {"question": "What is JSON?"},
                "outputs": {"answer": "JSON is a lightweight text format used to store and exchange data."},
            },
            {
                "inputs": {"question": "What is Docker?"},
                "outputs": {"answer": "Docker is a platform that packages applications into containers so they run the same everywhere."},
            },
            {
                "inputs": {"question": "What is a database?"},
                "outputs": {"answer": "A database is an organized system for storing and retrieving data."},
            },
            {
                "inputs": {"question": "What is machine learning?"},
                "outputs": {"answer": "Machine learning is a way for computers to learn patterns from data instead of being hard-coded with rules."},
            },
            {
                "inputs": {"question": "What is the capital of France?"},
                "outputs": {"answer": "The capital of France is Paris."},
            },
            {
                "inputs": {"question": "What is HTTP?"},
                "outputs": {"answer": "HTTP is the protocol browsers and servers use to request and send web pages."},
            },
        ],
    )

### Define Metrics (LLM As A Judge)


In [3]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

eval_llm = ChatOpenAI(model="gpt-5.4-mini-2026-03-17", temperature=0)

class CorrectnessGrade(BaseModel):
    reasoning: str = Field(
        description="Explain your reasoning for the score"
    )
    is_correct: bool = Field(
        description="True if the predicted answer matches the reference. Extra wording is OK. Contradictions are not."
    )

grader = eval_llm.with_structured_output(CorrectnessGrade)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    user_content = f"""QUESTION: {inputs["question"]}
GOLD: {reference_outputs["answer"]}
STUDENT: {outputs["response"]}"""
    grade = grader.invoke(
        [
            SystemMessage(content="Grade factual accuracy vs the gold answer. Accept paraphrases. Extra correct detail is OK. Mark incorrect only if the student contradicts the gold or gets the core fact wrong."),
            HumanMessage(content=user_content),
        ]
    )
    return {
        "key": "correctness",
        "score": 1.0 if grade.is_correct else 0.0,
        "comment": grade.reasoning,
    }

In [4]:
## Helpfulness — would this actually help a developer use the thing?
## Different axis from correctness: a true one-liner can still be useless.

class HelpfulnessGrade(BaseModel):
    reasoning: str = Field(description="Explain your reasoning for the score")
    is_helpful: bool = Field(
        description="True if a developer could act on this answer. Fail thin or generic replies even if they are factually true."
    )

helpfulness_grader = eval_llm.with_structured_output(HelpfulnessGrade)

def helpfulness(inputs: dict, outputs: dict) -> dict:
    user_content = f"""QUESTION: {inputs["question"]}
RESPONSE: {outputs["response"]}"""
    grade = helpfulness_grader.invoke(
        [
            SystemMessage(
                content="You are grading a developer chatbot. Score usefulness only. Ignore gold answers. A factually correct but too thin reply (e.g. 'a Python library') should fail."
            ),
            HumanMessage(content=user_content),
        ]
    )
    return {
        "key": "helpfulness",
        "score": 1.0 if grade.is_helpful else 0.0,
        "comment": grade.reasoning,
    }

### Run Evaluations

In [5]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

def my_app(question: str, model: str = "gpt-4o-mini", instructions: str = default_instructions) -> str:
    llm = ChatOpenAI(model=model, temperature=0)
    return llm.invoke(
        [
            SystemMessage(content=instructions),
            HumanMessage(content=question),
        ]
    ).content

In [6]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [7]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness, helpfulness],
    experiment_prefix="openai-4o-mini-chatbot"
)

View the evaluation results for experiment: 'openai-4o-mini-chatbot-4b0a4f30' at:
https://smith.langchain.com/o/0b65d61c-7187-5628-a4a9-83649180bab6/datasets/cf3ec8c5-3a4b-4cab-80b4-c544f4f2a56a/compare?selectedSessions=0f40c368-7d3e-4179-b4db-0436bbad4844




0it [00:00, ?it/s]

In [8]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"],model="gpt-4-turbo")}

In [9]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness, helpfulness],
    experiment_prefix="openai-4-turbo-chatbot"
)

View the evaluation results for experiment: 'openai-4-turbo-chatbot-d6d5667c' at:
https://smith.langchain.com/o/0b65d61c-7187-5628-a4a9-83649180bab6/datasets/cf3ec8c5-3a4b-4cab-80b4-c544f4f2a56a/compare?selectedSessions=53b5d133-5b1f-4bd3-b4d3-6fe702925d76




0it [00:00, ?it/s]

In [10]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"],model="gpt-5.4-mini-2026-03-17")}

In [11]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness, helpfulness],
    experiment_prefix="openai-gpt-5.4-mini-2026-03-17-chatbot"
)

View the evaluation results for experiment: 'openai-gpt-5.4-mini-2026-03-17-chatbot-c2f8c4c2' at:
https://smith.langchain.com/o/0b65d61c-7187-5628-a4a9-83649180bab6/datasets/cf3ec8c5-3a4b-4cab-80b4-c544f4f2a56a/compare?selectedSessions=93d5284c-2584-48f3-b364-eab0ffe2ceab




0it [00:00, ?it/s]

### Evaluation For RAG

In [12]:
## RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Add the document chunks to the "vector store" using OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings(),
)

# With langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=6)

/var/folders/nz/l_5d98c53hsgs16yvxmvz5q40000gn/T/ipykernel_27952/50547154.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [13]:
retriever.invoke("What are the components of an LLM agent?")

[Document(id='11ec92a8-1081-4302-9b19-9fcae3350f4f', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [14]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

# Judge stays fixed so model comparisons are fair
rag_llm = ChatOpenAI(model="gpt-5.4-mini-2026-03-17", temperature=0)
rag_llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15', 'langchain-openai': '1.4.3'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x1142eeba0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11493bb50>, root_client=<openai.OpenAI object at 0x11447fce0>, root_async_client=<openai.AsyncOpenAI object at 0x11447e030>, model_name='gpt-5.4-mini-2026-03-17', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True, stream_chunk_timeout=120.0)

In [15]:
from langsmith import traceable

@traceable()
def rag_bot(question: str, model: str = "gpt-4o-mini") -> dict:
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant. Answer using only the source documents.
If you don't know, say you don't know. Use three sentences maximum.

Documents:
{docs_string}"""

    llm = ChatOpenAI(model=model, temperature=0)
    ai_msg = llm.invoke(
        [
            SystemMessage(content=instructions),
            HumanMessage(content=question),
        ]
    )
    return {"answer": ai_msg.content, "documents": docs}

In [16]:
rag_bot("What are the components of an LLM agent?")

{'answer': 'The components of an LLM agent include Planning, Memory, and Tool Use. Planning involves task decomposition and self-reflection, while Memory encompasses different types of memory and Maximum Inner Product Search (MIPS). Tool Use involves providing the LLM with tools and instructions to effectively respond to user prompts.',
 'documents': [Document(id='11ec92a8-1081-4302-9b19-9fcae3350f4f', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brai

### Dataset

In [17]:
from langsmith import Client

client = Client()

# Questions the Lilian Weng posts can actually answer — gold is a full sentence
examples = [
    {
        "inputs": {"question": "How does ReAct combine reasoning and acting?"},
        "outputs": {"answer": "ReAct interleaves reasoning traces with actions such as a Wikipedia search, then observes the tool output before the next step."},
    },
    {
        "inputs": {"question": "What biases can appear with few-shot prompting?"},
        "outputs": {"answer": "Few-shot prompting can show majority label bias, recency bias, and common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks on LLMs?"},
        "outputs": {"answer": "Five types are token manipulation, gradient-based attacks, jailbreak prompting, human red-teaming, and model red-teaming."},
    },
    {
        "inputs": {"question": "What are the main components of an LLM-powered autonomous agent?"},
        "outputs": {"answer": "An LLM-powered agent uses the model as its brain, plus planning, memory, and tool use."},
    },
    {
        "inputs": {"question": "How do short-term and long-term memory differ in an LLM agent?"},
        "outputs": {"answer": "Short-term memory is in-context learning, while long-term memory stores information in an external vector store for later retrieval."},
    },
    {
        "inputs": {"question": "What is chain-of-thought prompting?"},
        "outputs": {"answer": "Chain-of-thought prompting asks the model to write intermediate reasoning steps before the final answer."},
    },
    {
        "inputs": {"question": "What is self-consistency in prompting?"},
        "outputs": {"answer": "Self-consistency samples several reasoning paths and selects the most consistent final answer."},
    },
    {
        "inputs": {"question": "What is prompt injection?"},
        "outputs": {"answer": "Prompt injection inserts malicious text that overrides the original instructions and hijacks the model's behavior."},
    },
    {
        "inputs": {"question": "What is Reflexion in LLM agents?"},
        "outputs": {"answer": "Reflexion lets an agent critique past actions, learn from mistakes, and refine the next steps."},
    },
    {
        "inputs": {"question": "What is PAL or program-aided language models?"},
        "outputs": {"answer": "PAL has the model write a program and use a code interpreter to compute the answer instead of doing the math in natural language."},
    },
]

dataset_name = "RAG Blog Evaluation"
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name=dataset_name)
    client.create_examples(dataset_id=dataset.id, examples=examples)

### Evaluators or Metrics
1. Correctness: Response vs reference answer
- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-judge to assess answer correctness.

In [18]:
from pydantic import BaseModel, Field

class CorrectnessGrade(BaseModel):
    reasoning: str = Field(description="Explain your reasoning for the score")
    is_correct: bool = Field(
        description="True if the student matches the gold facts. Extra correct detail is OK. Contradictions are not."
    )

correctness_grader = rag_llm.with_structured_output(CorrectnessGrade)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    user_content = f"""QUESTION: {inputs["question"]}
GOLD: {reference_outputs["answer"]}
STUDENT: {outputs["answer"]}"""
    grade = correctness_grader.invoke(
        [
            SystemMessage(
                content="Grade factual accuracy vs the gold answer. Accept paraphrases. Extra correct detail is OK. Mark incorrect only if the student contradicts the gold or gets the core fact wrong."
            ),
            HumanMessage(content=user_content),
        ]
    )
    return {
        "key": "correctness",
        "score": 1.0 if grade.is_correct else 0.0,
        "comment": grade.reasoning,
    }

### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [19]:
class RelevanceGrade(BaseModel):
    reasoning: str = Field(description="Explain your reasoning for the score")
    is_relevant: bool = Field(
        description="True if the answer addresses the question. Extra detail is OK. Off-topic replies are not."
    )

relevance_grader = rag_llm.with_structured_output(RelevanceGrade)

def relevance(inputs: dict, outputs: dict) -> dict:
    user_content = f"""QUESTION: {inputs["question"]}
STUDENT: {outputs["answer"]}"""
    grade = relevance_grader.invoke(
        [
            SystemMessage(
                content="Score whether the answer addresses the question. Ignore whether every fact is perfect."
            ),
            HumanMessage(content=user_content),
        ]
    )
    return {
        "key": "relevance",
        "score": 1.0 if grade.is_relevant else 0.0,
        "comment": grade.reasoning,
    }

### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [20]:
class GroundedGrade(BaseModel):
    reasoning: str = Field(description="Explain your reasoning for the score")
    is_grounded: bool = Field(
        description="True if every claim in the answer is supported by the retrieved documents."
    )

grounded_grader = rag_llm.with_structured_output(GroundedGrade)

def groundedness(inputs: dict, outputs: dict) -> dict:
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    user_content = f"""FACTS: {doc_string}
STUDENT: {outputs["answer"]}"""
    grade = grounded_grader.invoke(
        [
            SystemMessage(
                content="Score whether the answer is supported by the FACTS. Fail if it adds claims that are not in the documents."
            ),
            HumanMessage(content=user_content),
        ]
    )
    return {
        "key": "groundedness",
        "score": 1.0 if grade.is_grounded else 0.0,
        "comment": grade.reasoning,
    }

### Retrieval Relevance: Retrieved docs vs input

In [21]:
class RetrievalRelevanceGrade(BaseModel):
    reasoning: str = Field(description="Explain your reasoning for the score")
    is_relevant: bool = Field(
        description="True if the retrieved documents are related to the question. Some extra context is OK."
    )

retrieval_grader = rag_llm.with_structured_output(RetrievalRelevanceGrade)

def retrieval_relevance(inputs: dict, outputs: dict) -> dict:
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    user_content = f"""QUESTION: {inputs["question"]}
FACTS: {doc_string}"""
    grade = retrieval_grader.invoke(
        [
            SystemMessage(
                content="Score whether the retrieved FACTS are related to the QUESTION. Pass if they share any relevant meaning. Fail only if they are completely unrelated."
            ),
            HumanMessage(content=user_content),
        ]
    )
    return {
        "key": "retrieval_relevance",
        "score": 1.0 if grade.is_relevant else 0.0,
        "comment": grade.reasoning,
    }

### Run the evaluation

Same three models as the chatbot section. The judge (`rag_llm`) stays on gpt-5.4-mini so only the RAG generator changes.

In [22]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="openai-4o-mini-rag",
    metadata={"version": "gpt-4o-mini"},
)
experiment_results.to_pandas()

View the evaluation results for experiment: 'openai-4o-mini-rag-5295918d' at:
https://smith.langchain.com/o/0b65d61c-7187-5628-a4a9-83649180bab6/datasets/82992c98-fc6a-4c23-93ed-17d7c7d7ecaa/compare?selectedSessions=7bb52bf5-8229-4a55-aedb-04b7ba7fd072




0it [00:00, ?it/s]

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What biases can appear with few-shot prompting?,The biases that can appear with few-shot promp...,[page_content='Text: i'll bet the video game i...,None,Few-shot prompting can show majority label bia...,1.0,1.0,1.0,1.0,2.535818,0db03089-c9eb-4c26-81ea-c8299ca8fbf5,01a02714-1f64-77d1-914b-df8825beb1d4
1,What are five types of adversarial attacks on ...,The five types of adversarial attacks on LLMs ...,[page_content='Adversarial Attacks on LLMs | L...,None,"Five types are token manipulation, gradient-ba...",1.0,1.0,1.0,1.0,1.621424,175c6232-d67f-41d1-99c8-3b2b61810bca,01a02714-3c00-7a00-929b-14b7ec2c1548
2,What is Reflexion in LLM agents?,Reflexion is a framework that equips agents wi...,[page_content='Self-reflection is a vital aspe...,None,"Reflexion lets an agent critique past actions,...",1.0,1.0,1.0,1.0,1.922017,1ce5559c-f493-4a6d-911d-b056978150f0,01a02714-53c5-7b42-befa-8d87c54715af
3,What are the main components of an LLM-powered...,The main components of an LLM-powered autonomo...,[page_content='LLM Powered Autonomous Agents |...,None,An LLM-powered agent uses the model as its bra...,1.0,1.0,1.0,1.0,1.880546,4632b701-36f2-467a-8a43-98525d0a7871,01a02714-6f98-7282-8ed0-d7ca2b5508d2
4,How do short-term and long-term memory differ ...,Short-term memory in an LLM agent is used for ...,[page_content='Short-term memory: I would cons...,None,"Short-term memory is in-context learning, whil...",1.0,0.0,1.0,1.0,2.203181,51d62916-908d-451f-b961-9cac772f3263,01a02714-897a-7010-bd4c-98b210047f56
5,What is prompt injection?,I don't know.,[page_content='Automatic Prompt Design#\nPromp...,None,Prompt injection inserts malicious text that o...,0.0,0.0,0.0,1.0,1.515561,6bb1529e-a7db-479e-b62a-8e4eea9056df,01a02714-a985-7e23-8e81-026f2cdfdf93
6,What is chain-of-thought prompting?,Chain-of-thought (CoT) prompting is a techniqu...,[page_content='[8] Wang et al. “Self-Consisten...,None,Chain-of-thought prompting asks the model to w...,1.0,1.0,1.0,1.0,1.693983,76bab690-7612-458a-8c79-2da5ef0b5148,01a02714-bfe2-7f13-9f63-6b866a8b4685
7,How does ReAct combine reasoning and acting?,ReAct combines reasoning and acting by extendi...,[page_content='Self-reflection is a vital aspe...,None,ReAct interleaves reasoning traces with action...,1.0,1.0,1.0,1.0,2.894842,7abb3941-74c7-4d16-b0e8-8b8bff64a681,01a02714-d9bf-72a1-85c6-d55a2d58941d
8,What is PAL or program-aided language models?,"PAL, or Program-aided language models, is a fr...",[page_content='[19] Zhou et al. “Large Languag...,None,PAL has the model write a program and use a co...,1.0,0.0,1.0,1.0,2.180276,8cf9128e-ca31-4804-b9cd-1072ef6a18de,01a02714-f69d-7ce2-b74c-5674a7dba608
9,What is self-consistency in prompting?,Self-consistency in prompting refers to a meth...,[page_content='[8] Wang et al. “Self-Consisten...,None,Self-consistency samples several reasoning pat...,1.0,1.0,1.0,1.0,2.175540,b33ca649-f610-48a2-8d2a-eeec60b2bf64,01a02715-12f0-7be3-8319-955431e4ea22


In [23]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"], model="gpt-4-turbo")

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="openai-4-turbo-rag",
    metadata={"version": "gpt-4-turbo"},
)
experiment_results.to_pandas()

View the evaluation results for experiment: 'openai-4-turbo-rag-0f52126c' at:
https://smith.langchain.com/o/0b65d61c-7187-5628-a4a9-83649180bab6/datasets/82992c98-fc6a-4c23-93ed-17d7c7d7ecaa/compare?selectedSessions=84d1e2f1-44d2-4f8c-8e62-c464e1e77fea




0it [00:00, ?it/s]

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What biases can appear with few-shot prompting?,"In few-shot prompting, several biases can appe...",[page_content='Text: i'll bet the video game i...,None,Few-shot prompting can show majority label bia...,1.0,1.0,1.0,1.0,4.752737,0db03089-c9eb-4c26-81ea-c8299ca8fbf5,01a02715-3eea-7472-8b95-3cd554f93d71
1,What are five types of adversarial attacks on ...,The five types of adversarial attacks on LLMs ...,[page_content='Adversarial Attacks on LLMs | L...,None,"Five types are token manipulation, gradient-ba...",1.0,1.0,1.0,1.0,3.633706,175c6232-d67f-41d1-99c8-3b2b61810bca,01a02715-63c4-7ea1-94f2-fea5a7833128
2,What is Reflexion in LLM agents?,"Reflexion, as described by Shinn & Labash (202...",[page_content='Self-reflection is a vital aspe...,None,"Reflexion lets an agent critique past actions,...",1.0,1.0,1.0,1.0,5.493411,1ce5559c-f493-4a6d-911d-b056978150f0,01a02715-8659-75f3-8caa-ecc2ae4af0ef
3,What are the main components of an LLM-powered...,The main components of an LLM-powered autonomo...,[page_content='LLM Powered Autonomous Agents |...,None,An LLM-powered agent uses the model as its bra...,1.0,1.0,1.0,1.0,5.625375,4632b701-36f2-467a-8a43-98525d0a7871,01a02715-b3b4-7d20-8f17-27bbab68f23f
4,How do short-term and long-term memory differ ...,"In an LLM agent, short-term memory is used to ...",[page_content='Short-term memory: I would cons...,None,"Short-term memory is in-context learning, whil...",1.0,0.0,1.0,1.0,3.955180,51d62916-908d-451f-b961-9cac772f3263,01a02715-da13-7dd2-8621-babddc62b057
5,What is prompt injection?,I don't know.,[page_content='Automatic Prompt Design#\nPromp...,None,Prompt injection inserts malicious text that o...,0.0,0.0,0.0,1.0,1.788893,6bb1529e-a7db-479e-b62a-8e4eea9056df,01a02715-ffad-7a01-8204-b27512464896
6,What is chain-of-thought prompting?,Chain of thought (CoT) prompting is a techniqu...,[page_content='[8] Wang et al. “Self-Consisten...,None,Chain-of-thought prompting asks the model to w...,1.0,1.0,1.0,1.0,3.129992,76bab690-7612-458a-8c79-2da5ef0b5148,01a02716-1d48-7851-8b50-747535559b6a
7,How does ReAct combine reasoning and acting?,ReAct combines reasoning and acting by extendi...,[page_content='Self-reflection is a vital aspe...,None,ReAct interleaves reasoning traces with action...,1.0,1.0,1.0,1.0,4.410973,7abb3941-74c7-4d16-b0e8-8b8bff64a681,01a02716-3f18-7a92-8fef-797935f5a8a8
8,What is PAL or program-aided language models?,"PAL, or Program-aided Language Models, refers ...",[page_content='[19] Zhou et al. “Large Languag...,None,PAL has the model write a program and use a co...,1.0,0.0,1.0,1.0,4.823512,8cf9128e-ca31-4804-b9cd-1072ef6a18de,01a02716-6715-7a62-b8a1-6c8ed74ba4c5
9,What is self-consistency in prompting?,"I don't have specific information on ""self-con...",[page_content='[8] Wang et al. “Self-Consisten...,None,Self-consistency samples several reasoning pat...,0.0,1.0,0.0,1.0,2.394942,b33ca649-f610-48a2-8d2a-eeec60b2bf64,01a02716-8e68-78e3-9802-040f9a4b9267


In [24]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"], model="gpt-5.4-mini-2026-03-17")

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="openai-gpt-5.4-mini-2026-03-17-rag",
    metadata={"version": "gpt-5.4-mini-2026-03-17"},
)
experiment_results.to_pandas()

View the evaluation results for experiment: 'openai-gpt-5.4-mini-2026-03-17-rag-bbcaf5a7' at:
https://smith.langchain.com/o/0b65d61c-7187-5628-a4a9-83649180bab6/datasets/82992c98-fc6a-4c23-93ed-17d7c7d7ecaa/compare?selectedSessions=3577f0ab-cca3-4fa8-8df9-46ea95fc05e5




0it [00:00, ?it/s]

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What biases can appear with few-shot prompting?,The document says few-shot prompting can be af...,[page_content='Text: i'll bet the video game i...,None,Few-shot prompting can show majority label bia...,1.0,1.0,1.0,1.0,1.723010,0db03089-c9eb-4c26-81ea-c8299ca8fbf5,01a02716-b1e0-7041-ad3f-16ffd7ef20f9
1,What are five types of adversarial attacks on ...,The document says there are five approaches: t...,[page_content='Adversarial Attacks on LLMs | L...,None,"Five types are token manipulation, gradient-ba...",1.0,1.0,1.0,1.0,1.598868,175c6232-d67f-41d1-99c8-3b2b61810bca,01a02716-cbc0-70d0-8380-67b1ab3cf4b5
2,What is Reflexion in LLM agents?,Reflexion is a framework that equips LLM agent...,[page_content='Self-reflection is a vital aspe...,None,"Reflexion lets an agent critique past actions,...",1.0,1.0,1.0,1.0,1.969965,1ce5559c-f493-4a6d-911d-b056978150f0,01a02716-e3f8-7210-a866-61ae2c6c49ab
3,What are the main components of an LLM-powered...,"The main components are planning, memory, and ...",[page_content='LLM Powered Autonomous Agents |...,None,An LLM-powered agent uses the model as its bra...,1.0,1.0,1.0,1.0,3.406335,4632b701-36f2-467a-8a43-98525d0a7871,01a02717-11d1-7ac0-992e-a6fc2473ba96
4,How do short-term and long-term memory differ ...,Short-term memory is the in-context learning t...,[page_content='Short-term memory: I would cons...,None,"Short-term memory is in-context learning, whil...",1.0,1.0,1.0,1.0,4.148202,51d62916-908d-451f-b961-9cac772f3263,01a02717-45d1-7681-872c-ca6d7691c5da
5,What is prompt injection?,I don’t know.,[page_content='Automatic Prompt Design#\nPromp...,None,Prompt injection inserts malicious text that o...,0.0,1.0,0.0,1.0,2.996771,6bb1529e-a7db-479e-b62a-8e4eea9056df,01a02717-81c5-7993-90b2-dc3bef1d1840
6,What is chain-of-thought prompting?,Chain-of-thought (CoT) prompting is a techniqu...,[page_content='[8] Wang et al. “Self-Consisten...,None,Chain-of-thought prompting asks the model to w...,1.0,1.0,1.0,1.0,3.529246,76bab690-7612-458a-8c79-2da5ef0b5148,01a02717-b5c1-7ff2-951f-d1fda2f7051a
7,How does ReAct combine reasoning and acting?,ReAct combines reasoning and acting by extendi...,[page_content='Self-reflection is a vital aspe...,None,ReAct interleaves reasoning traces with action...,1.0,1.0,1.0,1.0,3.492188,7abb3941-74c7-4d16-b0e8-8b8bff64a681,01a02717-e8b7-76a1-821e-afa08a30694d
8,What is PAL or program-aided language models?,"PAL, or Program-aided Language Models, is a me...",[page_content='[19] Zhou et al. “Large Languag...,None,PAL has the model write a program and use a co...,1.0,1.0,1.0,1.0,3.627171,8cf9128e-ca31-4804-b9cd-1072ef6a18de,01a02718-21d3-7271-ba48-5cd27b172c5f
9,What is self-consistency in prompting?,Self-consistency is a method that improves cha...,[page_content='[8] Wang et al. “Self-Consisten...,None,Self-consistency samples several reasoning pat...,1.0,0.0,1.0,1.0,3.651820,b33ca649-f610-48a2-8d2a-eeec60b2bf64,01a02718-569e-74e1-bcd6-ff228599d6df


In [ ]:
# LangSmith's dataset page often shows the scale max (1), not the mean.
# Use the table you already have: some rows are 0.
df = experiment_results.to_pandas()
score_cols = [c for c in df.columns if c.startswith("feedback.")]
print(df[score_cols].mean().round(2))
print()
print(df[["inputs.question"] + score_cols])